# Importação de Bibliotecas

In [ ]:
# Controle de Arquivos
import os

# Computação Numérica
import numpy as np

# Manipulação e Visualização de Dados
import pandas as pd
from datasets import Dataset, DatasetDict

# Aprendizado de Máquina
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix, classification_report
from imblearn.under_sampling import RandomUnderSampler
from transformers import AutoTokenizer, AutoModel, AutoModelForSequenceClassification, DataCollatorWithPadding, TrainingArguments, Trainer
import torch
import inspect

## Caminhos de Arquivos

In [ ]:
# Altere os caminhos conforme necessário

# Corpus SteamBR filtrado (arquivo pickle)
path_steamcorpusbr_filtrado = "df_pt.pkl"

# Dicionário Léxico LIWC (pt-br)
path_liwc = "Brazilian_Portuguese_LIWC2015_dictionary.dic"

# Dicionário de Características de Produto
path_product_features = "product_features.txt"

# Subconjuntos de Saída
path_output_dev = "symbolic_dev_df.pkl"
path_output_train = "symbolic_train_df.pkl"
path_output_test = "symbolic_test_df.pkl"

# Pré-Processamento de Dataset Filtrado

## Importação

In [ ]:
df = pd.read_pickle(path_steamcorpusbr_filtrado)

## Classificação de Referência

In [ ]:
T = 0.5
df['weighted_vote_score'] = df['weighted_vote_score'].astype("float32")
df["labels"] = (df["weighted_vote_score"] > T).astype(int)

187730
15221
labels
1    187730
0     15221
Name: count, dtype: int64


## Divisão da Base de Dados

In [ ]:
# Treinamento (70%) and Temporário (30%):
train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)

# Desenvolvimento (10%) e Teste (20%):
dev_df, test_df = train_test_split(temp_df, test_size=2/3, random_state=42)

## Subamostragem da Base de Dados

In [ ]:
rus = RandomUnderSampler(random_state=42)

X_resampled, y_resampled = rus.fit_resample(
  train_df[["review"]],
  train_df["labels"]
)

train_df = X_resampled.copy()
train_df["labels"] = y_resampled

print(train_df["labels"].value_counts())

## Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('neuralmind/bert-base-portuguese-cased', do_lower_case=False)

def tokenize(batch):
  return tokenizer(
    batch["review"],
    truncation=True,
    max_length=128
  )

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


## Importação do Modelo

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
  "neuralmind/bert-base-portuguese-cased",
  num_labels=2
)

model.config.id2label = {
  0: "not_useful",
  1: "useful"
}

model.config.label2id = {
  "not_useful": 0,
  "useful": 1
}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from th

## Informações do Modelo

In [ ]:
inspect.getsource(model.forward)

    @can_return_tuple
    @auto_docstring
    def forward(
        self,
        input_ids: torch.Tensor | None = None,
        attention_mask: torch.Tensor | None = None,
        token_type_ids: torch.Tensor | None = None,
        position_ids: torch.Tensor | None = None,
        inputs_embeds: torch.Tensor | None = None,
        labels: torch.Tensor | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple[torch.Tensor] | SequenceClassifierOutput:
        r"""
        labels (`torch.LongTensor` of shape `(batch_size,)`, *optional*):
            Labels for computing the sequence classification/regression loss. Indices should be in `[0, ...,
            config.num_labels - 1]`. If `config.num_labels == 1` a regression loss is computed (Mean-Square loss), If
            `config.num_labels > 1` a classification loss is computed (Cross-Entropy).
        """
        outputs = self.bert(
            input_ids,
            attention_mask=attention_mask,
            token_ty

## Criar DatasetDict

In [ ]:
dataset = DatasetDict({
  'train': Dataset.from_pandas(train_df),
  'dev': Dataset.from_pandas(dev_df),
  'test': Dataset.from_pandas(test_df)
})

## Tokenização

In [ ]:
tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(['review'])

tokenized_dataset.set_format(
  type='torch',
  columns=['input_ids', 'attention_mask', 'labels']
)

Map:   0%|          | 0/21196 [00:00<?, ? examples/s]

Map:   0%|          | 0/20295 [00:00<?, ? examples/s]

Map:   0%|          | 0/40591 [00:00<?, ? examples/s]

{'train': ['labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'], 'dev': ['recommendationid', 'language', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'review_id', 'game', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_playtime_forever', 'author_playtime_last_two_weeks', 'author_playtime_at_review', 'author_last_played', 'timestamp_dev_responded', 'developer_response', 'votes_log', 'is_pt', 'labels', '__index_level_0__', 'input_ids', 'token_type_ids', 'attention_mask'], 'test': ['recommendationid', 'language', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'review_id', 'game', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author

## Hiperparâmetros do Modelo

In [ ]:
os.makedirs("./logs", exist_ok=True)

training_args = TrainingArguments(
  output_dir="./results",
  num_train_epochs=3,
  learning_rate=2e-5,
  per_device_train_batch_size=16,
  per_device_eval_batch_size=64,
  warmup_steps=1000,
  weight_decay=0.01,

  eval_strategy="epoch",
  save_strategy="epoch",

  logging_strategy="steps",
  logging_steps=1000,
  logging_dir="./logs",

  load_best_model_at_end=True,
  metric_for_best_model="f1",
  save_total_limit=2,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = logits.argmax(axis=1)
  return {
    "accuracy": accuracy_score(labels, preds),
    "precision": precision_score(labels, preds),
    "recall": recall_score(labels, preds),
    "f1": f1_score(labels, preds)
  }

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
  model=model,
  args=training_args,
  train_dataset=tokenized_dataset["train"],
  eval_dataset=tokenized_dataset["dev"],
  compute_metrics=compute_metrics,
  data_collator=data_collator
)

In [ ]:
if not torch.cuda.is_available():
  print("Aviso: Não há uma GPU ativada nesta sessão. O treinamento vai demorar muito tempo.")

SyntaxError: invalid syntax (943679918.py, line 1)

# Treinamento do Modelo

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.612510,0.589518,0.762602,0.962852,0.772829,0.857439
2,0.538701,0.556291,0.773639,0.965592,0.782857,0.864675
3,0.478998,0.636336,0.742942,0.965527,0.748453,0.843244


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3975, training_loss=0.5097673140231919, metrics={'train_runtime': 1985.4379, 'train_samples_per_second': 32.027, 'train_steps_per_second': 2.002, 'total_flos': 4177932225840960.0, 'train_loss': 0.5097673140231919, 'epoch': 3.0})

# Avaliação do Modelo

In [ ]:
trainer.evaluate()

Training Loss,Validation Loss,Epoch,Accuracy,Precision,Recall,F1
0.478998,0.556291,3,0.773639,0.965592,0.782857,0.864675


{'eval_loss': 0.5562906861305237,
 'eval_accuracy': 0.7736388272973639,
 'eval_precision': 0.9655921052631579,
 'eval_recall': 0.7828568380627267,
 'eval_f1': 0.8646753858842936}

In [ ]:
trainer.state.best_model_checkpoint

'./results/checkpoint-2650'

In [ ]:
predictions = trainer.predict(tokenized_dataset["dev"])

import numpy as np

preds = np.argmax(predictions.predictions, axis=1)
labels = predictions.label_ids

In [ ]:
classification_report(labels, preds)

              precision    recall  f1-score   support

           0       0.20      0.66      0.31      1547
           1       0.97      0.78      0.86     18748

    accuracy                           0.77     20295
   macro avg       0.58      0.72      0.59     20295
weighted avg       0.91      0.77      0.82     20295



In [ ]:
import numpy as np

predictions = trainer.predict(tokenized_dataset["dev"])
preds = np.argmax(predictions.predictions, axis=1)

np.unique(preds, return_counts=True)

(array([0, 1]), array([ 5095, 15200]))

In [ ]:
confusion_matrix(predictions.label_ids, preds)

[[ 1024   523]
 [ 4071 14677]]


In [ ]:
df = pd.read_pickle("symbolic_test_df.pkl")

In [ ]:
predictions = trainer.predict(tokenized_dataset["test"])

In [ ]:
import numpy as np

predictions = trainer.predict(tokenized_dataset["test"])

df_test = dataset["test"].to_pandas()

df_test["pred"] = np.argmax(predictions.predictions, axis=1)

df_test["pred_label"] = df_test["pred"].map({
    0: "not_useful",
    1: "useful"
})

In [ ]:
from scipy.special import softmax

probs = softmax(predictions.predictions, axis=1)

df_test["prob_not_useful"] = probs[:, 0]
df_test["prob_useful"] = probs[:, 1]

In [ ]:
df_test.to_pickle("subsymbolic_test_df.pkl")